In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
df = pd.read_csv("LoansDatasest.csv")
df.head()

,customer_id,customer_age,customer_income,home_ownership,employment_duration,loan_intent,loan_grade,loan_amnt,loan_int_rate,term_years,historical_default,cred_hist_length,Current_loan_status
0,1.0,22,59000,RENT,123.0,PERSONAL,C,"£35,000.00",16.02,10,Y,3,DEFAULT
1,2.0,21,9600,OWN,5.0,EDUCATION,A,"£1,000.00",11.14,1,NaN,2,NO DEFAULT
2,3.0,25,9600,MORTGAGE,1.0,MEDICAL,B,"£5,500.00",12.87,5,N,3,DEFAULT
3,4.0,23,65500,RENT,4.0,MEDICAL,B,"£35,000.00",15.23,10,N,2,DEFAULT
4,5.0,24,54400,RENT,8.0,MEDICAL,B,"£35,000.00",14.27,10,Y,4,DEFAULT


In [4]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

Shape: (32586, 13)

Columns:
Index(['customer_id', 'customer_age', 'customer_income', 'home_ownership',
       'employment_duration', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'term_years', 'historical_default', 'cred_hist_length',
       'Current_loan_status'],
      dtype='object')

Data Types:
customer_id            float64
customer_age             int64
customer_income         object
home_ownership          object
employment_duration    float64
loan_intent             object
loan_grade              object
loan_amnt               object
loan_int_rate          float64
term_years               int64
historical_default      object
cred_hist_length         int64
Current_loan_status     object
dtype: object


In [5]:
df.isnull().sum()

customer_id                3
customer_age               0
customer_income            0
home_ownership             0
employment_duration      895
loan_intent                0
loan_grade                 0
loan_amnt                  1
loan_int_rate           3116
term_years                 0
historical_default     20737
cred_hist_length           0
Current_loan_status        4
dtype: int64

In [6]:
# Remove customer_id
df = df.drop("customer_id", axis=1)

# Change customer_income
df["customer_income"] = df["customer_income"].replace(',', '', regex=True)
df["customer_income"] = df["customer_income"].astype(float)

# Chnage loan_amnt
df["loan_amnt"] = df["loan_amnt"].replace('[£,]', '', regex=True)
df["loan_amnt"] = df["loan_amnt"].astype(float)

# Convert GBP → INR
df["loan_amnt"] = df["loan_amnt"] * 105

In [7]:
df[["customer_income","loan_amnt"]].head()

,customer_income,loan_amnt
0,59000.0,3675000.0
1,9600.0,105000.0
2,9600.0,577500.0
3,65500.0,3675000.0
4,54400.0,3675000.0


In [8]:
df = df.dropna(subset=["Current_loan_status"])

In [9]:
df["employment_duration"] = df["employment_duration"].fillna(df["employment_duration"].median())

df["loan_int_rate"] = df["loan_int_rate"].fillna(df["loan_int_rate"].median())

df["loan_amnt"] = df["loan_amnt"].fillna(df["loan_amnt"].median())

In [10]:
df["historical_default"] = df["historical_default"].fillna("No")

In [11]:
df.isnull().sum()

customer_age           0
customer_income        0
home_ownership         0
employment_duration    0
loan_intent            0
loan_grade             0
loan_amnt              0
loan_int_rate          0
term_years             0
historical_default     0
cred_hist_length       0
Current_loan_status    0
dtype: int64

In [12]:
df["Current_loan_status"] = df["Current_loan_status"].map({
    "NO DEFAULT":0,
    "DEFAULT":1
})

In [13]:
df["Current_loan_status"].value_counts()

Current_loan_status
0    25742
1     6840
Name: count, dtype: int64

In [14]:
df["historical_default"] = df["historical_default"].map({
    "Yes":1,
    "No":0
})

In [15]:
df = pd.get_dummies(df, columns=[
    "home_ownership",
    "loan_intent",
    "loan_grade"
])

In [16]:
df.head()

,customer_age,customer_income,employment_duration,loan_amnt,loan_int_rate,term_years,historical_default,cred_hist_length,Current_loan_status,home_ownership_MORTGAGE,...,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_A,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E
0,22,59000.0,123.0,3675000.0,16.02,10,NaN,3,1,False,...,False,False,False,True,False,False,False,True,False,False
1,21,9600.0,5.0,105000.0,11.14,1,0.0,2,0,False,...,True,False,False,False,False,True,False,False,False,False
2,25,9600.0,1.0,577500.0,12.87,5,NaN,3,1,True,...,False,False,True,False,False,False,True,False,False,False
3,23,65500.0,4.0,3675000.0,15.23,10,NaN,2,1,False,...,False,False,True,False,False,False,True,False,False,False
4,24,54400.0,8.0,3675000.0,14.27,10,NaN,4,1,False,...,False,False,True,False,False,False,True,False,False,False


In [17]:
df.shape

(32582, 24)

In [18]:
df["historical_default"] = df["historical_default"].fillna(0)

In [19]:
df["historical_default"].value_counts()

historical_default
0.0    32582
Name: count, dtype: int64

In [20]:
X = df.drop("Current_loan_status", axis=1)
y = df["Current_loan_status"]

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (26065, 23)
Testing Data: (6517, 23)


In [22]:
import warnings
warnings.filterwarnings("ignore")

In [23]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [24]:
y_pred = model.predict(X_test)

In [25]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8364278042043886

Confusion Matrix:
[[4932  218]
 [ 848  519]]


In [26]:
df["predicted_default"] = model.predict(X)

In [60]:
# Take user input
age = int(input("Enter customer age: "))
income = float(input("Enter customer income: "))
employment = float(input("Enter employment duration (years): "))
loan_amount = float(input("Enter loan amount (INR): "))
interest = float(input("Enter loan interest rate: "))
term = int(input("Enter loan term (years): "))
history = int(input("Historical default? (0 = No, 1 = Yes): "))
cred_length = int(input("Credit history length: "))

# Create a sample with same structure
user_data = X.iloc[[0]].copy()

# Fill numeric fields
user_data.loc[:, ["customer_age","customer_income","employment_duration",
                  "loan_amnt","loan_int_rate","term_years",
                  "historical_default","cred_hist_length"]] = [
                  age,income,employment,loan_amount,interest,term,history,cred_length]

# Predict
prediction = model.predict(user_data)

# Show result
if prediction[0] == 0:
    print("\nPrediction: Loan is SAFE (No Default Expected)")
else:
    print("\nPrediction: High Risk of DEFAULT")

Enter customer age:  20
Enter customer income:  66000
Enter employment duration (years):  0
Enter loan amount (INR):  350000
Enter loan interest rate:  8
Enter loan term (years):  15
Historical default? (0 = No, 1 = Yes):  0
Credit history length:  0



Prediction: Loan is SAFE (No Default Expected)


In [64]:
# Take user input
age = int(input("Enter customer age: "))
income = float(input("Enter customer income: "))
employment = float(input("Enter employment duration (years): "))
loan_amount = float(input("Enter loan amount (INR): "))
interest = float(input("Enter loan interest rate: "))
term = int(input("Enter loan term (years): "))
history = int(input("Historical default? (0 = No, 1 = Yes): "))
cred_length = int(input("Credit history length: "))

# Create a sample with same structure
user_data = X.iloc[[0]].copy()

# Fill numeric fields
user_data.loc[:, ["customer_age","customer_income","employment_duration",
                  "loan_amnt","loan_int_rate","term_years",
                  "historical_default","cred_hist_length"]] = [
                  age,income,employment,loan_amount,interest,term,history,cred_length]

# Predict
prediction = model.predict(user_data)

# Show result
if prediction[0] == 0:
    print("\nPrediction: Loan is SAFE (No Default Expected)")
else:
    print("\nPrediction: High Risk of DEFAULT")

Enter customer age:  19
Enter customer income:  12000
Enter employment duration (years):  0
Enter loan amount (INR):  2500000
Enter loan interest rate:  20
Enter loan term (years):  7
Historical default? (0 = No, 1 = Yes):  0
Credit history length:  0



Prediction: High Risk of DEFAULT


In [27]:
import pickle
pickle.dump(model, open("credit_risk_model.pkl", "wb"))